# Word2Vec ile duygu analizi

### Alıştırma hedefleri:
- Word2Vec ile kelimeleri vektörlere dönüştürmek
- Word2vec tarafından verilen kelime temsilini RNN'ye beslemek için kullanmak

<hr>

▶️ Bu hücreyi çalıştırın ve kullandığınız 📚 [Gensim - Word2Vec](https://radimrehurek.com/gensim/auto_examples/index.html) sürümünün ≥ 4.0 olduğundan emin olun!

In [1]:
!pip freeze | grep gensim

gensim==4.4.0


# Veri


❓ **Soru** ❓ Öncelikle verileri yükleyelim. Fonksiyonda neler olduğunu anlamanıza gerek yok, burada önemi yok.

⚠️ **Uyarı** ⚠️ `load_data` fonksiyonunda `percentage_of_sentences` argümanı vardır. Bilgisayarınıza bağlı olarak, çok fazla cümle bilgisayarınızı yavaşlatabilir veya hatta dondurabilir - RAM'iniz taşabilir. Bu nedenle, **cümlelerin %10'uyla başlamalı** ve bilgisayarınızın bunu kaldırabildiğini kontrol etmelisiniz. Aksi takdirde, daha düşük bir sayı ile yeniden çalıştırın. 

⚠️ **DISCLAIMER** ⚠️ **_En büyüğü kimde_ (_who has the biggest_)(RAM) oyununu oynamaya gerek yok!** Buradaki amaç, modellerinizi hızlı bir şekilde çalıştırarak prototip oluşturmaktır. Gerçek hayatta bile, hızlı bir şekilde döngü ve hata ayıklama yapmak için verilerinizin bir alt kümesiyle başlamanız önerilir. Bu nedenle, yalnızca en iyi doğruluğu elde etmek istiyorsanız sayıyı artırın.

In [2]:
import numpy as np

In [3]:
####################################################
### Verileri yüklemek için bu hücreyi çalıştırın ###
####################################################

import tensorflow_datasets as tfds
from tensorflow.keras.preprocessing.text import text_to_word_sequence

def load_data(percentage_of_sentences=None):
    train_data, test_data = tfds.load(name="imdb_reviews", split=["train", "test"], batch_size=-1, as_supervised=True)

    train_sentences, y_train = tfds.as_numpy(train_data)
    test_sentences, y_test = tfds.as_numpy(test_data)

    # Tüm verilerin yalnızca belirli bir yüzdesini alın
    if percentage_of_sentences is not None:
        assert(percentage_of_sentences> 0 and percentage_of_sentences<=100)

        len_train = int(percentage_of_sentences/100*len(train_sentences))
        train_sentences, y_train = train_sentences[:len_train], y_train[:len_train]

        len_test = int(percentage_of_sentences/100*len(test_sentences))
        test_sentences, y_test = test_sentences[:len_test], y_test[:len_test]

    X_train = [text_to_word_sequence(_.decode("utf-8")) for _ in train_sentences]
    X_test = [text_to_word_sequence(_.decode("utf-8")) for _ in test_sentences]

    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = load_data(percentage_of_sentences=10)

I0000 00:00:1780469007.391013   24582 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780469007.391636   24582 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780469007.469441   24582 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780469008.825324   24582 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

Önceki alıştırmada, Word2vec temsilini eğittiniz ve bu Şekil'in ilk adımında gösterildiği gibi, tüm eğitim cümlelerinizi bir RNN'ye beslemek için dönüştürdünüz: 

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/06-DL/NLP/word2vec_representation.png" alt="Word2Vec with RNN" width="400px" />



❓ **Soru** ❓ Burada, önceki alıştırmada yaptığınızın aynısını tekrar yapalım. İlk olarak, eğitim cümleniz üzerinde bir word2vec modeli (istediğiniz argümanlarla) eğitin. Bunu `word2vec` değişkenine kaydedin.

In [4]:
import sys
print(sys.executable)

/home/doruk/.pyenv/versions/workintech/bin/python


In [5]:
from gensim.models import Word2Vec

word2vec = Word2Vec(
    sentences=X_train,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1
)

Önceki alıştırmadaki işlevleri yeniden kullanarak, eğitim ve test verilerinizi RNN'ye girebileceğiniz bir biçime dönüştürelim.

❓ **Soru** ❓ Neler olduğunu anladığınızdan emin olmak için aşağıdaki işlevi okuyun ve çalıştırın.

In [6]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Bir cümleyi (kelime listesi) gömme uzayındaki kelimeleri temsil eden bir matrise dönüştürme işlevi
def embed_sentence(word2vec, sentence):
    embedded_sentence = []
    for word in sentence:
        if word in word2vec.wv:
            embedded_sentence.append(word2vec.wv[word])

    return np.array(embedded_sentence)

# Bir cümle listesini matris listesine dönüştüren işlev
def embedding(word2vec, sentences):
    embed = []

    for sentence in sentences:
        embedded_sentence = embed_sentence(word2vec, sentence)
        embed.append(embedded_sentence)

    return embed

# Eğitim ve test cümlelerini gömün(embed edin)
X_train_embed = embedding(word2vec, X_train)
X_test_embed = embedding(word2vec, X_test)


# Eğitim ve test gömülü cümleleri doldurun
X_train_pad = pad_sequences(X_train_embed, dtype='float32', padding='post', maxlen=200)
X_test_pad = pad_sequences(X_test_embed, dtype='float32', padding='post', maxlen=200)

☝️ Çalıştığından emin olmak için, `X_train_pad` ve `X_test_pad` için aşağıdakileri kontrol edelim:

- bunlar numpy dizileridir
- bunlar 3 boyutludur
- son boyut, word2vec gömme alanınızın boyutundadır (bunu `word2vec.wv.vector_size` ile elde edebilirsiniz
- ilk boyut, `X_train` ve `X_test` boyutundadır

✅ **İyi Uygulama** ✅ Bu tür testler oldukça önemlidir! Sadece bu alıştırmada değil, gerçek hayattaki uygulamalarda da. Hataları çok geç fark etmeyi ve bunların tüm not defterine yayılmasını önler.

In [7]:
# BENİ TEST ET
for X in [X_train_pad, X_test_pad]:
    assert type(X) == np.ndarray
    assert X.shape[-1] == word2vec.wv.vector_size


assert X_train_pad.shape[0] == len(X_train)
assert X_test_pad.shape[0] == len(X_test)

# Temel model

Kendi modelinizi test etmek için çok basit bir modele sahip olmak her zaman iyidir - çok basit bir algoritmadan daha iyi bir şey yaptığınızdan emin olmak için.

❓ **Soru** ❓ Temel doğruluk oranınız nedir? Bu durumda, temeliniz `y_train` içinde en çok bulunan etiketi tahmin etmek olabilir (tabii ki, veri kümesi dengeli ise, temel doğruluk oranı 1/n'dir; burada n, sınıfların sayısıdır - burada 2'dir).

In [8]:
import numpy as np

values, counts = np.unique(y_train, return_counts=True)
baseline_accuracy = counts.max() / counts.sum()

baseline_accuracy

0.506

# Model

❓ **Soru** ❓ Aşağıdaki katmanlara sahip bir RNN yazın:
- bir `Masking` katmanı
- 20 birim ve `tanh` aktivasyon fonksiyonuna sahip bir `LSTM`
- 10 birimlik bir `Dense`
- görevinize bağlı bir çıktı katmanı

Ardından, modelinizi derleyin (en azından başlangıçta optimizer olarak `rmsprop` kullanmanızı öneririz).

In [9]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Masking, LSTM, Dense

model = Sequential([
    Masking(mask_value=0.0, input_shape=(200, word2vec.wv.vector_size)),
    LSTM(20, activation="tanh"),
    Dense(10, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()# SENİN KODUN BURAYA

/home/doruk/.pyenv/versions/workintech/lib/python3.12/site-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking (Masking)               │ (None, 200, 100)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 20)             │         9,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │           210 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,901 (38.68 KB)

 Trainable params: 9,901 (38.68 KB)

 Non-trainable params: 0 (0.00 B)

❓ **Soru** ❓ Modeli gömülü ve doldurulmuş verilerinize uyarlayın - erken durdurma kriterini unutmayın.

❗ **Not** ❗ Doğruluğunuz büyük ölçüde eğitim kümenize bağlı olacaktır. Burada, performansınızın temel modelin üzerinde olduğundan emin olun (bu, başlangıçtaki IMDB verilerinin yalnızca %20'sini yüklemiş olsanız bile geçerli olmalıdır).

In [10]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/20


E0000 00:00:1780469170.636254   24582 util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 74ms/step - accuracy: 0.5320 - loss: 0.6917 - val_accuracy: 0.6240 - val_loss: 0.6809
Epoch 2/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 60ms/step - accuracy: 0.5935 - loss: 0.6757 - val_accuracy: 0.6600 - val_loss: 0.6379
Epoch 3/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 58ms/step - accuracy: 0.6245 - loss: 0.6506 - val_accuracy: 0.5560 - val_loss: 0.7373
Epoch 4/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 58ms/step - accuracy: 0.6210 - loss: 0.6527 - val_accuracy: 0.6620 - val_loss: 0.6309
Epoch 5/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 61ms/step - accuracy: 0.6335 - loss: 0.6369 - val_accuracy: 0.7100 - val_loss: 0.6077
Epoch 6/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 61ms/step - accuracy: 0.6580 - loss: 0.6303 - val_accuracy: 0.6680 - val_loss: 0.6087
Epoch 7/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 62ms/step - accuracy: 0.6500 - loss: 0.6254 - val_accuracy: 0.7100 - val_loss: 0.5788
Epoch 8/20
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 63ms/step - accuracy: 0.6610 - loss: 0.6155 - val_accuracy: 0.6980 - val_loss: 0.

❓ **Soru** ❓ Test setinde modelinizi değerlendirin

In [11]:
loss, accuracy = model.evaluate(X_test_pad, y_test)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6992 - loss: 0.5829
Test Loss: 0.5829
Test Accuracy: 0.6992


# Eğitimli Word2Vec - Transfer Öğrenimi

Doğruluğunuz, temel modelin üzerinde olsa da, oldukça düşük olabilir. Veri temizleme ve gömme kalitesini iyileştirme gibi bunu iyileştirmek için birçok seçenek vardır.

Burada veri temizleme stratejilerine girmeyeceğiz. Gömme kalitemizi iyileştirmeye çalışalım. Ancak, daha büyük bir metin kümesini yüklemek yerine, neden başkalarının öğrendiği gömme modelinden yararlanmayalım? Çünkü gömme modelinin kalitesi, yani kelimelerin yakınlığı, farklı görevlerden elde edilebilir. Transfer öğrenme tam olarak budur.

❓ **Soru** ❓ Bu sayede word2vec'te bulunan tüm farklı modelleri listeleyin: 

In [12]:
import gensim.downloader as api
print(list(api.info()['models'].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


ℹ️ Modellerin listesini ve boyutlarını [`gensim-data` deposunda](https://github.com/RaRe-Technologies/gensim-data#models) de bulabilirsiniz.

❓ **Soru** ❓ Önceden eğitilmiş word2vec gömme alanlarından birini yükleyin.

Bunu `api.load(seçtiğiniz model)` ile yapabilir ve `word2vec_transfer` içinde saklayabilirsiniz.

<details>
    <summary>💡 İpucu</summary>
    
`glove-wiki-gigaword-50` modeli, daha küçük olması (65 MB) nedeniyle başlangıç için iyi bir seçenektir.

</details>

In [13]:
import gensim.downloader as api

word2vec_transfer = api.load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded


❓ **Soru** ❓ Kelime dağarcığının boyutunu ve aynı zamanda gömme alanının boyutunu kontrol edin.

In [15]:
vocab_size = len(word2vec_transfer.index_to_key)
embedding_dim = word2vec_transfer.vector_size

print(vocab_size)
print(embedding_dim)

400000
50


❓ İlk soruda yaptığımız gibi, `X_train` ve `X_test`'i gömelim! (`embed_sentence_with_TF` işlevinde küçük bir fark var, ancak bu konuya girmeyeceğiz.)

In [16]:
# Bir cümleyi (kelime listesi) gömme uzayındaki kelimeleri temsil eden bir matrise dönüştürme işlevi
def embed_sentence_with_TF(word2vec, sentence):
    embedded_sentence = []
    for word in sentence:
        if word in word2vec:
            embedded_sentence.append(word2vec[word])

    return np.array(embedded_sentence)

# Bir cümle listesini matris listesine dönüştüren işlev
def embedding(word2vec, sentences):
    embed = []

    for sentence in sentences:
        embedded_sentence = embed_sentence_with_TF(word2vec, sentence)
        embed.append(embedded_sentence)

    return embed

# Eğitim ve test cümlelerini gömün (embed edin)
X_train_embed_2 = embedding(word2vec_transfer, X_train)
X_test_embed_2 = embedding(word2vec_transfer, X_test)

❓ **Soru** ❓  Sonuçlarınızı doldurmayı ve bunları `X_train_pad_2` ve `X_test_pad_2` içinde saklamayı unutmayın.

In [17]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_pad_2 = pad_sequences(
    X_train_embed_2,
    dtype="float32",
    padding="post",
    maxlen=200
)

X_test_pad_2 = pad_sequences(
    X_test_embed_2,
    dtype="float32",
    padding="post",
    maxlen=200
)

print(X_train_pad_2.shape)
print(X_test_pad_2.shape)

(2500, 200, 50)
(2500, 200, 50)


❓ **Soru** ❓ Modeli yeniden başlatın ve yeni gömülü (ve doldurulmuş(padded)) verilerinize uyarlayın!  Test setinizde değerlendirin ve önceki doğruluğunuzla karşılaştırın.

❗ **Not** ❗ Buradaki eğitim biraz zaman alabilir. Sadece 10 dönem hesaplayabilir (bu **iyi** bir uygulama değildir, sadece çok uzun süre beklememek içindir) ve eğitim sürerken bir sonraki alıştırmaya geçebilir veya bir mola verebilirsiniz, muhtemelen bunu hak ettiniz ;)

In [18]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Masking, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

model_transfer = Sequential([
    Masking(mask_value=0.0, input_shape=(200, word2vec_transfer.vector_size)),
    LSTM(20, activation="tanh"),
    Dense(10, activation="relu"),
    Dense(1, activation="sigmoid")
])

model_transfer.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history_transfer = model_transfer.fit(
    X_train_pad_2,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

loss, accuracy = model_transfer.evaluate(X_test_pad_2, y_test)

print(f"Transfer Test Loss: {loss:.4f}")
print(f"Transfer Test Accuracy: {accuracy:.4f}")

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 64ms/step - accuracy: 0.5200 - loss: 0.6931 - val_accuracy: 0.5680 - val_loss: 0.6888
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 61ms/step - accuracy: 0.5820 - loss: 0.6804 - val_accuracy: 0.6100 - val_loss: 0.6719
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 62ms/step - accuracy: 0.6285 - loss: 0.6518 - val_accuracy: 0.6800 - val_loss: 0.6099
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 67ms/step - accuracy: 0.6730 - loss: 0.6133 - val_accuracy: 0.6800 - val_loss: 0.6008
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 77ms/step - accuracy: 0.6955 - loss: 0.5876 - val_accuracy: 0.6880 - val_loss: 0.5831
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 76ms/step - accuracy: 0.7120 - loss: 0.5756 - val_accuracy: 0.6920 - val_loss: 0.5867
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 74ms/step - accuracy: 0.7075 - loss: 0.5705 - val_accuracy: 0.7100 - val_loss: 0.5590
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 71ms/step - accuracy: 0.7370 - loss: 0.5464 - val_accuracy: 0.7020 - v

In [20]:
res = model_transfer.evaluate(X_test_pad_2, y_test, verbose=0)

print(f"The accuracy evaluated on the test set is of {res[1]*100:.3f}%")

The accuracy evaluated on the test set is of 73.560%


Yeni word2vec'iniz büyük bir metin kümesinde eğitildiğinden, çok sayıda kelimeyi temsil eder! Küçük veri kümenize kıyasla çok daha fazladır, özellikle de eğitim kümesinde belirli bir sayıdan fazla bulunmayan kelimeleri elediğiniz için. Bu nedenle, eğitim ve test kümenizde çok daha fazla gömülü kelime vardır, bu da her yinelemeyi öncekinden daha uzun hale getirir.